In [ ]:
# 1. Install dependencies
!pip install transformers datasets accelerate huggingface_hub peft "torchao>=0.16.0"

# 2. Login to Hugging Face
from huggingface_hub import login
login("YOUR_HF_TOKEN_HERE") # Replace with your token


In [ ]:
import os
import sys
if not os.path.exists('adelic-spectral-zeta'):
    !git clone https://github.com/sneed-and-feed/adelic-spectral-zeta.git
sys.path.insert(0, os.path.abspath('adelic-spectral-zeta'))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from src.ultrametric_v3.llama_patcher import inject_surgery
from src.ultrametric_v3.surgery_trainer import TauAnnealingCallback, SurgeryTrainer

model_id = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")

# Patch the model
model = inject_surgery(model)

# WRAP LLAMA'S BRAIN IN LORA!
# We want the base model to learn how to route around the severed connections!
from peft import get_peft_model, LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)

# Make sure our custom routers are STILL trainable alongside the LoRA adapters!
for name, param in model.named_parameters():
    if "router" in name:
        param.requires_grad = True

model.print_trainable_parameters()

# Load the FULL dataset (not just 1%)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True).remove_columns(["text"])
tokenized_datasets.set_format("torch")

# The Real Run!
training_args = TrainingArguments(
    output_dir="./surgery_results_final",
    per_device_train_batch_size=2,
    learning_rate=1e-3, # High learning rate is safe because ONLY the routers are training!
    max_steps=1000,     # Run for 1000 steps so the penalty can slowly carve out the sparse topology
    logging_steps=50,
    save_steps=2000,    # Don't freeze the notebook to save checkpoints!
)

from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = SurgeryTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_datasets,
    callbacks=[TauAnnealingCallback(initial_tau=1.0, min_tau=0.1, decay_steps=1000)],
    surgery_lambda_init=0.0,
    surgery_lambda_max=0.5,  # Turn up the pressure to prune!
    surgery_ramp_steps=800   # Slowly ramp up the penalty over 500 steps
)

trainer.train()


In [ ]:
# Run Inference to see the Pruning Results!
model.eval()

total_heads = 0
pruned_heads = 0
for i, layer in enumerate(model.model.layers):
    router_z = layer.self_attn.router.z
    layer_pruned = (router_z > 0).sum().item() # If z > 0, g_h evaluates to 1.0 (Sparse Mask Applied!)
    pruned_heads += layer_pruned
    total_heads += len(router_z)
    
print(f"\nSurgery Result: The model successfully carved out a sparse topology, permanently pruning {pruned_heads}/{total_heads} attention heads!")

prompt = "The spectral zeta function of the p-adic numbers is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)

print("\n--- GENERATION ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
